# Zurich Airpot data cleaning

In [46]:
# Import necessary libraries
import pandas as pd
import glob
import  matplotlib.pyplot as plt
from pathlib import Path

In [47]:
# Load and concatenate all CSV files from the specified folder
file_paths = glob.glob("./data/zurich*.csv")
df = pd.concat((pd.read_csv(file) for file in file_paths), ignore_index=True)
df

,Date,Type,LocalAirport,ForeignAirport,Airline,Status,Planned,Expected
0,2024-10-28,arrival,Zurich,Geneva,swiss,Landed,21:05,20:58
1,2024-10-28,arrival,Zurich,Berlin,swiss,Landed,21:05,21:03
2,2024-10-28,arrival,Zurich,Paris CDG,swiss,Landed,21:10,21:07
3,2024-10-28,arrival,Zurich,Venice,swiss,Landed,21:15,21:07
4,2024-10-28,arrival,Zurich,Brussels,swiss,Landed,21:20,21:16
...,...,...,...,...,...,...,...,...
9111,2024-11-13,departure,Zurich,Pristina,edelweiss,Departed,06:45,06:40
9112,2024-11-13,departure,Zurich,London LHR,british,Departed,06:50,NaN
9113,2024-11-13,departure,Zurich,Amsterdam,klm,Departed,06:55,NaN
9114,2024-11-13,departure,Zurich,Belgrade,swiss,Departed,06:55,07:10


In [48]:
# For better understanding let's rename the column Expected to Actual
df.rename(columns={'Expected': 'Actual'}, inplace=True)

In [49]:
# Fill NaN values in Expected column with values from Planned
df['Actual'].isnull().sum()

np.int64(2336)

In [50]:
df['Actual'] = df['Actual'].fillna(df['Planned']) # these are flights that are simply on time
df['Actual'].isnull().sum()

np.int64(0)

In [51]:
# Convert Planned and Actual columns to datetime format for calculations
df['PlannedDate'] = pd.to_datetime(df['Date'] + " " + df['Planned'])
df['ActualDate'] = pd.to_datetime(df['Date'] + " " + df['Actual'])
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values(by='Date', ascending=True).head(5)

,Date,Type,LocalAirport,ForeignAirport,Airline,Status,Planned,Actual,PlannedDate,ActualDate
842,2024-10-21,arrival,Zurich,Copenhagen,swiss,Landed,11:30,11:53,2024-10-21 11:30:00,2024-10-21 11:53:00
876,2024-10-21,arrival,Zurich,Malaga,swiss,Landed,09:10,09:18,2024-10-21 09:10:00,2024-10-21 09:18:00
875,2024-10-21,arrival,Zurich,Kos,edelweiss,Landed,12:45,13:05,2024-10-21 12:45:00,2024-10-21 13:05:00
874,2024-10-21,arrival,Zurich,Heraklion,edelweiss,Landed,12:40,13:03,2024-10-21 12:40:00,2024-10-21 13:03:00
873,2024-10-21,arrival,Zurich,Munich,lufthansa_neu,Landed,12:40,13:07,2024-10-21 12:40:00,2024-10-21 13:07:00


In [52]:
# Add Delay column for further analysis
df['Delay'] = ((df['ActualDate'] - df['PlannedDate'])
                              .dt.total_seconds() / 60)

In [53]:
# Remove unnecessary columns
columns_to_remove = ['Actual', 'Planned', 'Date','Status']

# Drop columns if they exist
df_last_two_weeks = df.drop(columns=[col for col in columns_to_remove if col in df.columns])
df_last_two_weeks

,Type,LocalAirport,ForeignAirport,Airline,PlannedDate,ActualDate,Delay
0,arrival,Zurich,Geneva,swiss,2024-10-28 21:05:00,2024-10-28 20:58:00,-7.0
1,arrival,Zurich,Berlin,swiss,2024-10-28 21:05:00,2024-10-28 21:03:00,-2.0
2,arrival,Zurich,Paris CDG,swiss,2024-10-28 21:10:00,2024-10-28 21:07:00,-3.0
3,arrival,Zurich,Venice,swiss,2024-10-28 21:15:00,2024-10-28 21:07:00,-8.0
4,arrival,Zurich,Brussels,swiss,2024-10-28 21:20:00,2024-10-28 21:16:00,-4.0
...,...,...,...,...,...,...,...
9111,departure,Zurich,Pristina,edelweiss,2024-11-13 06:45:00,2024-11-13 06:40:00,-5.0
9112,departure,Zurich,London LHR,british,2024-11-13 06:50:00,2024-11-13 06:50:00,0.0
9113,departure,Zurich,Amsterdam,klm,2024-11-13 06:55:00,2024-11-13 06:55:00,0.0
9114,departure,Zurich,Belgrade,swiss,2024-11-13 06:55:00,2024-11-13 07:10:00,15.0


In [56]:
# last check for NaN
print(df.isna().sum())

Date              0
Type              0
LocalAirport      0
ForeignAirport    0
Airline           0
Status            0
Planned           0
Actual            0
PlannedDate       0
ActualDate        0
Delay             0
dtype: int64


In [54]:
# Save the final dataset
file_path = Path(f'data/combined_zurich_airport.csv')
# Create the directory if it doesn't exist
file_path.parent.mkdir(parents=True, exist_ok=True)
# Save and overwrite if it already exists
df.to_csv(file_path, header=True, index=False)